In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

mkdir: cannot create directory ‘/root/.kaggle’: File exists
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
# !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

In [ ]:

# !unzip chest-xray-pneumonia.zip -d chest-xray-pneumonia

In [ ]:

import  tensorflow as tf
from tensorflow import keras
import cv2
import numpy as np

In [ ]:
path=r'/content/chest-xray-pneumonia/chest_xray/train/NORMAL/IM-0117-0001.jpeg'
image=cv2.imread(path)
np.max(image)

np.uint8(255)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
datagen=ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.3,
    horizontal_flip=True,
    shear_range=0.2
)
train_ds=datagen.flow_from_directory(
    '/content/chest-xray-pneumonia/chest_xray/train',
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

Found 5216 images belonging to 2 classes.


In [ ]:
# train_ds=keras.utils.image_dataset_from_directory(
#     directory=r'/content/chest-xray-pneumonia/chest_xray/train',
#     labels='inferred',
#     label_mode='int',
#     color_mode='rgb',
#     batch_size=16, # Reduced batch size
#     image_size=(150,150)
# )
test_ds=keras.utils.image_dataset_from_directory(
    directory=r'/content/chest-xray-pneumonia/chest_xray/test',
    labels='inferred',
    label_mode='int',
    color_mode='rgb',
    batch_size=16, # Reduced batch size
    image_size=(224,224)
)
val_ds=keras.utils.image_dataset_from_directory(
    directory=r'/content/chest-xray-pneumonia/chest_xray/val',
    labels='inferred',
    label_mode='int',
    color_mode='rgb',
    batch_size=16, # Reduced batch size
    image_size=(224,224)
)

Found 624 files belonging to 2 classes.
Found 16 files belonging to 2 classes.


In [ ]:
# rescale form 0 - 1
def process(image,label):
   image=tf.cast(image/255.0,tf.float32)
   return image,label

In [ ]:
# train_ds=train_ds.map(process)
test_ds=test_ds.map(process)
val_ds=val_ds.map(process)

In [ ]:
# Get the shape and data type for a single image in your dataset
print(val_ds.element_spec)


(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))


# MODEL BUILDING

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D,Dense,Flatten,MaxPooling2D,BatchNormalization,Dropout,GlobalAveragePooling2D
from keras import layers, regularizers
from tensorflow.keras.applications.resnet50 import ResNet50


In [ ]:
conv_base=ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [ ]:
for layer in conv_base.layers:
  print(layer.name)

input_layer_6
conv1_pad
conv1_conv
conv1_bn
conv1_relu
pool1_pad
pool1_pool
conv2_block1_1_conv
conv2_block1_1_bn
conv2_block1_1_relu
conv2_block1_2_conv
conv2_block1_2_bn
conv2_block1_2_relu
conv2_block1_0_conv
conv2_block1_3_conv
conv2_block1_0_bn
conv2_block1_3_bn
conv2_block1_add
conv2_block1_out
conv2_block2_1_conv
conv2_block2_1_bn
conv2_block2_1_relu
conv2_block2_2_conv
conv2_block2_2_bn
conv2_block2_2_relu
conv2_block2_3_conv
conv2_block2_3_bn
conv2_block2_add
conv2_block2_out
conv2_block3_1_conv
conv2_block3_1_bn
conv2_block3_1_relu
conv2_block3_2_conv
conv2_block3_2_bn
conv2_block3_2_relu
conv2_block3_3_conv
conv2_block3_3_bn
conv2_block3_add
conv2_block3_out
conv3_block1_1_conv
conv3_block1_1_bn
conv3_block1_1_relu
conv3_block1_2_conv
conv3_block1_2_bn
conv3_block1_2_relu
conv3_block1_0_conv
conv3_block1_3_conv
conv3_block1_0_bn
conv3_block1_3_bn
conv3_block1_add
conv3_block1_out
conv3_block2_1_conv
conv3_block2_1_bn
conv3_block2_1_relu
conv3_block2_2_conv
conv3_block2_2_bn


In [ ]:


conv_base.trainable = True

set_trainable = False

for layer in conv_base.layers:
  if layer.name == 'conv5_block3_2_relu':
    set_trainable = True
  if set_trainable:
    layer.trainable = True
  else:
    layer.trainable = False

for layer in conv_base.layers:
  print(layer.name,layer.trainable)


input_layer_6 False
conv1_pad False
conv1_conv False
conv1_bn False
conv1_relu False
pool1_pad False
pool1_pool False
conv2_block1_1_conv False
conv2_block1_1_bn False
conv2_block1_1_relu False
conv2_block1_2_conv False
conv2_block1_2_bn False
conv2_block1_2_relu False
conv2_block1_0_conv False
conv2_block1_3_conv False
conv2_block1_0_bn False
conv2_block1_3_bn False
conv2_block1_add False
conv2_block1_out False
conv2_block2_1_conv False
conv2_block2_1_bn False
conv2_block2_1_relu False
conv2_block2_2_conv False
conv2_block2_2_bn False
conv2_block2_2_relu False
conv2_block2_3_conv False
conv2_block2_3_bn False
conv2_block2_add False
conv2_block2_out False
conv2_block3_1_conv False
conv2_block3_1_bn False
conv2_block3_1_relu False
conv2_block3_2_conv False
conv2_block3_2_bn False
conv2_block3_2_relu False
conv2_block3_3_conv False
conv2_block3_3_bn False
conv2_block3_add False
conv2_block3_out False
conv3_block1_1_conv False
conv3_block1_1_bn False
conv3_block1_1_relu False
conv3_block1

In [ ]:

conv_base.summary()

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_6[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 1,054,720 (4.02 MB)

 Non-trainable params: 22,532,992 (85.96 MB)

In [ ]:
model=Sequential()
model.add(conv_base)
model.add(GlobalAveragePooling2D())

model.add(Dense(256,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,113,537 (91.99 MB)

 Trainable params: 1,580,033 (6.03 MB)

 Non-trainable params: 22,533,504 (85.96 MB)

In [ ]:
from keras.optimizers import Adam

In [ ]:
model.compile(loss='binary_crossentropy',metrics=['accuracy'],optimizer=Adam(1e-5))


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

# Add reduce_lr to your callbacks list in model.fit

In [ ]:
model.fit(train_ds,validation_data=test_ds,epochs=30,callbacks=reduce_lr,class_weight={0: 2.0, 1: 1.0} )

Epoch 1/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 129s 358ms/step - accuracy: 0.7719 - loss: 0.5966 - val_accuracy: 0.8077 - val_loss: 0.5884 - learning_rate: 1.0000e-05
Epoch 2/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 109s 335ms/step - accuracy: 0.8144 - loss: 0.5003 - val_accuracy: 0.8510 - val_loss: 0.3657 - learning_rate: 1.0000e-05
Epoch 3/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 112s 344ms/step - accuracy: 0.8269 - loss: 0.4794 - val_accuracy: 0.8510 - val_loss: 0.3497 - learning_rate: 1.0000e-05
Epoch 4/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 111s 340ms/step - accuracy: 0.8378 - loss: 0.4622 - val_accuracy: 0.6619 - val_loss: 0.7290 - learning_rate: 1.0000e-05
Epoch 5/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 108s 330ms/step - accuracy: 0.8468 - loss: 0.4327 - val_accuracy: 0.8381 - val_loss: 0.3753 - learning_rate: 1.0000e-05
Epoch 6/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 110s 336ms/step - accuracy: 0.8572 - loss: 0.4211 - val_accuracy: 0.8718 - val_loss: 0.3353 - learning_rate: 2.0000e-06
Epoch 7/30
326/326 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
predict=model.predict(test_ds)



In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:

import numpy as np

# Get true labels from test_ds
y_true = []
for _, labels in test_ds:
    y_true.extend(labels.numpy())
y_true = np.array(y_true)

# Convert predictions (probabilities) to binary labels
y_pred = (predict > 0.5).astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
import pickle
with open('model.pkl','wb') as f:
  pickle.dump(model,f)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
sns.heatmap(confusion_matrix(y_true,y_pred),annot=True,fmt='.2f')